# K-Nearest Neighbor

Data yang dipakai berisi hasil pemeriksaan laboratorium sampel tumor. Tugasnya mengklasifikasikan tumor sebagai jinak atau ganas.

## Tujuan

- Melatih model KNN dengan jumlah tetangga berbeda.
- Mengevaluasi kinerja model klasifikasi.
- Menentukan jumlah tetangga terbaik.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

RANDOM_STATE = 123


## Memuat data tumor


In [ ]:
DATASET_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/tumor.csv"
)
tumor_data = pd.read_csv(DATASET_URL)
tumor_data.head()


In [ ]:
tumor_data.columns


Setiap baris memuat ciri sel tumor, misalnya kepadatan dan bentuk inti sel. Kolom terakhir adalah label kelas.


In [ ]:
features = tumor_data.iloc[:, :-1]
target = tumor_data.iloc[:, -1].squeeze()
features.describe()


Seluruh fitur bernilai numerik pada rentang 1 sampai 10, sehingga tidak perlu penstandaran tambahan.


In [ ]:
target.value_counts(normalize=True)


In [ ]:
target.value_counts().plot.bar(color=["green", "red"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


Sekitar 65 persen sampel jinak dan 35 persen ganas, jadi sebaran kelasnya masih wajar.


## Membagi data latih dan uji


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
)


## KNN dengan dua tetangga


In [ ]:
neighbor_model = KNeighborsClassifier(n_neighbors=2)
neighbor_model.fit(X_train, y_train)
predictions = neighbor_model.predict(X_test)


In [ ]:
def evaluate_metrics(actual, predicted):
    precision, recall, f_beta, _ = precision_recall_fscore_support(
        actual, predicted, average="binary"
    )
    return {
        "accuracy": accuracy_score(actual, predicted),
        "recall": recall,
        "precision": precision,
        "f1score": f_beta,
    }


evaluate_metrics(y_test, predictions)


Model sudah mengenali tumor ganas dengan sangat baik. Selanjutnya jumlah tetangga diubah menjadi lima.


## Latihan: KNN dengan lima tetangga


In [ ]:
five_neighbor_model = KNeighborsClassifier(n_neighbors=5)
five_neighbor_model.fit(X_train, y_train)
five_predictions = five_neighbor_model.predict(X_test)
evaluate_metrics(y_test, five_predictions)


## Menentukan jumlah tetangga terbaik

Nilai K diuji dari 1 sampai 50, lalu F1 Score setiap K disimpan dan digambarkan.


In [ ]:
max_k = 50

f1_results = pd.DataFrame(
    [
        {
            "K": k,
            "F1 Score": f1_score(
                y_test,
                KNeighborsClassifier(n_neighbors=k)
                .fit(X_train, y_train)
                .predict(X_test),
            ),
        }
        for k in range(1, max_k + 1)
    ]
).set_index("K")
f1_results.head()


In [ ]:
axis = f1_results.plot(figsize=(10, 6))
axis.set(xlabel="Num of Neighbors", ylabel="F1 Score")
axis.set_xticks(range(1, max_k, 2))
plt.ylim((0.85, 1))
plt.title("KNN F1 Score")
plt.tight_layout()
plt.show()


Kurva menunjukkan F1 Score tertinggi berada di sekitar K sama dengan 5. Nilai K yang terlalu kecil membuat model peka terhadap derau, sedangkan K yang terlalu besar memperhalus batas kelas.


## Ringkasan

KNN bekerja baik pada data tumor ini karena skala fiturnya sudah seragam. Pemilihan jumlah tetangga lewat kurva F1 memberi hasil yang lebih dapat dipertanggungjawabkan daripada menebak nilainya.
